# Incremental Dry-Run on Existing Downloads

This notebook scans existing files in `Raw playlists:tracks/R&B` and `Raw playlists:tracks/Rock`, maps them to `track_id` using `sampled_spotify_tracks.csv`, builds a temporary `track_id` mirror with symlinks, and runs a downloader dry-run to validate incremental skip behavior.

In [ ]:
from pathlib import Path
import os
import re
import pandas as pd

from src.data.deemix_pipeline import download_tracks_from_csv

In [ ]:
PROJECT_ROOT = Path('.').resolve()
RAW_ROOT = Path('/Users/gabriel/Documents/Spring 2026/Neural Networks/Raw playlists:tracks')
EXISTING_DIRS = [RAW_ROOT / 'R&B', RAW_ROOT / 'Rock']

CATALOG_CSV = PROJECT_ROOT / 'sampled_spotify_tracks.csv'
MANIFEST_CSV = PROJECT_ROOT / 'data' / 'processed' / 'existing_tracks_manifest.csv'
DRYRUN_INPUT_CSV = PROJECT_ROOT / 'data' / 'processed' / 'existing_tracks_dryrun_input.csv'
MIRROR_DIR = PROJECT_ROOT / 'data' / 'processed' / 'existing_trackid_mirror'
DRYRUN_OUT_CSV = PROJECT_ROOT / 'data' / 'processed' / 'existing_tracks_dryrun_results.csv'

for p in [MANIFEST_CSV.parent, MIRROR_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, RAW_ROOT, EXISTING_DIRS

In [ ]:
def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def parse_artist_title_from_filename(path: Path):
    stem = path.stem
    if ' - ' in stem:
        artist, title = stem.split(' - ', 1)
    else:
        artist, title = '', stem
    return artist.strip(), title.strip()

In [ ]:
# 1) Inventory existing audio files in the two source folders
rows = []
audio_exts = {'.mp3', '.flac', '.m4a', '.wav'}
for folder in EXISTING_DIRS:
    for p in sorted(folder.glob('*')):
        if p.is_file() and p.suffix.lower() in audio_exts:
            artist_guess, title_guess = parse_artist_title_from_filename(p)
            rows.append({
                'source_folder': folder.name,
                'download_filepath': str(p.resolve()),
                'artist_guess': artist_guess,
                'title_guess': title_guess,
                'artist_norm': normalize_text(artist_guess),
                'title_norm': normalize_text(title_guess),
            })

existing_df = pd.DataFrame(rows)
len(existing_df), existing_df.head(3)

In [ ]:
# 2) Load catalog and map existing files -> track_id
catalog = pd.read_csv(CATALOG_CSV).copy()
catalog['artist_primary'] = catalog['artists'].astype(str).str.split(';').str[0]
catalog['artist_norm'] = catalog['artist_primary'].apply(normalize_text)
catalog['title_norm'] = catalog['track_name'].apply(normalize_text)

mapped = existing_df.merge(
    catalog[['track_id', 'artists', 'track_name', 'track_genre', 'artist_norm', 'title_norm']],
    on=['artist_norm', 'title_norm'],
    how='left',
)

mapped['matched'] = mapped['track_id'].notna()
mapped['track_id'] = mapped['track_id'].astype(str).replace({'nan': ''})
mapped.to_csv(MANIFEST_CSV, index=False)

mapped['matched'].value_counts(dropna=False), MANIFEST_CSV

In [ ]:
# 3) Build dry-run input CSV from matched rows only
dry_input = mapped[mapped['track_id'].astype(str).str.len() > 0][['track_id', 'artists', 'track_name', 'download_filepath', 'source_folder']].copy()
dry_input = dry_input.drop_duplicates(subset=['track_id']).reset_index(drop=True)
dry_input.to_csv(DRYRUN_INPUT_CSV, index=False)
len(dry_input), DRYRUN_INPUT_CSV

In [ ]:
# 4) Create track_id-based mirror dir using symlinks (no file copy)
created = 0
already = 0
skipped_missing = 0
for _, row in dry_input.iterrows():
    src = Path(row['download_filepath'])
    if not src.exists():
        skipped_missing += 1
        continue
    tid = str(row['track_id']).strip()
    dst = MIRROR_DIR / f'{tid}{src.suffix.lower()}'
    if dst.exists() or dst.is_symlink():
        already += 1
        continue
    os.symlink(src, dst)
    created += 1

{'created_symlinks': created, 'already_present': already, 'missing_source': skipped_missing}

In [ ]:
# 5) Incremental dry-run: should mostly report already_downloaded
dryrun_df = download_tracks_from_csv(
    DRYRUN_INPUT_CSV,
    csv_out=DRYRUN_OUT_CSV,
    output_dir=MIRROR_DIR,
    deemix_binary='python -m deemix',
    output_flag='-p',
    target='spotify_url',
    dedupe_on='track_id',
    check_existing=True,
    dry_run=True,
    verbose=False,
    save_every=100,
)

dryrun_df['deemix_status'].value_counts(dropna=False), DRYRUN_OUT_CSV

In [ ]:
dryrun_df[['track_id', 'deemix_status', 'download_filepath', 'immutable_key']].head(20)